
# Exp6 — Pooled-Metric Reanalysis

The full Exp6 run completed correctly, but the first summary notebook reported the **mean of target-wise percentage gains**:

\[
\frac{1}{N}\sum_i 100\frac{\mathrm{MSE}^{\rm Shared}_i-\mathrm{MSE}^{\rm Adaptive}_i}{\mathrm{MSE}^{\rm Shared}_i}.
\]

The manuscript's primary controlled-selection metric is instead the **pooled endpoint-MSE gain**:

\[
G_{\rm adapt}(H)=100\frac{\sum_i \mathrm{MSE}^{\rm Shared}_i(H)-\sum_i \mathrm{MSE}^{\rm Adaptive}_i(H)}{\sum_i \mathrm{MSE}^{\rm Shared}_i(H)}.
\]

This notebook **does not rerun any experiment**. It reloads the already-saved Exp6 row-level CSV files and recomputes the robustness analysis with the manuscript-matched pooled metric.

The target-wise mean is retained only as a secondary diagnostic.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("results_highdim_subset_cap_robustness")
TARGET_SEEDS = [2026, 2027, 2028, 2029, 2030]
DATASETS = ["Electricity", "PeMS04"]
EXPECTED_POOLED_GAIN = {
    ("Electricity", 96): -0.0445,
    ("Electricity", 192): 1.4992,
    ("Electricity", 336): -2.1514,
    ("Electricity", 720): 0.6137,
    ("PeMS04", 12): -0.5556,
    ("PeMS04", 24): -3.0974,
    ("PeMS04", 48): 28.9567,
}
print("ROOT:", ROOT)
print("Exists:", ROOT.exists())


In [ ]:
files = sorted(ROOT.glob("*_rows.csv"))
if not files:
    raise FileNotFoundError("No saved Exp6 row-level CSV files found. Run the FULL Exp6 notebook first.")
frames = []
needed = {"dataset","target_strategy","target_seed","candidate_cap","target_channel","horizon","shared_test_mse","adaptive_test_mse","adaptive_gain_vs_shared_%"}
for p in files:
    df = pd.read_csv(p)
    if needed.issubset(df.columns):
        frames.append(df)
if not frames:
    raise FileNotFoundError("No compatible Exp6 row-level CSV files found.")
all_rows = pd.concat(frames, ignore_index=True)
print("Loaded rows:", len(all_rows))


## 1. Manuscript-matched pooled condition metric


In [ ]:
def pooled_gain(group):
    shared = float(group["shared_test_mse"].sum())
    adaptive = float(group["adaptive_test_mse"].sum())
    return 100.0 * (shared - adaptive) / max(shared, 1e-12)

keys = ["dataset", "target_strategy", "target_seed", "candidate_cap", "horizon"]
rows = []
for key, g in all_rows.groupby(keys):
    d, strategy, seed, cap, h = key
    rows.append({
        "dataset": d, "target_strategy": strategy, "target_seed": int(seed),
        "candidate_cap": int(cap), "horizon": int(h),
        "n_targets": int(g["target_channel"].nunique()),
        "pooled_gain_pct": pooled_gain(g),
        "mean_targetwise_gain_pct": float(g["adaptive_gain_vs_shared_%"].mean()),
        "median_targetwise_gain_pct": float(g["adaptive_gain_vs_shared_%"].median()),
        "target_win_fraction": float((g["adaptive_gain_vs_shared_%"] > 0).mean()),
        "sum_shared_mse": float(g["shared_test_mse"].sum()),
        "sum_adaptive_mse": float(g["adaptive_test_mse"].sum()),
    })
pooled = pd.DataFrame(rows).sort_values(keys).reset_index(drop=True)
pooled.to_csv(ROOT / "full_POOLED_condition_summary.csv", index=False)
display(pooled.round(4))


## 2. Exact-original reproduction check


In [ ]:
orig = pooled[(pooled["target_strategy"] == "even") & (pooled["candidate_cap"] == 128)].copy()
orig["expected_manuscript_pooled_gain_pct"] = [EXPECTED_POOLED_GAIN.get((d, int(h)), np.nan) for d, h in zip(orig["dataset"], orig["horizon"])]
orig["abs_difference_pctpt"] = np.abs(orig["pooled_gain_pct"] - orig["expected_manuscript_pooled_gain_pct"])
orig.to_csv(ROOT / "full_POOLED_original_reproduction_check.csv", index=False)
display(orig.round(4))
mx = float(orig["abs_difference_pctpt"].max())
print("Max absolute difference:", round(mx, 6), "percentage points")
print("PASS" if mx <= 0.10 else "WARNING: investigate before interpretation")


## 3. Random-target robustness — primary pooled metric


In [ ]:
rand = pooled[(pooled["target_strategy"] == "random") & (pooled["candidate_cap"] == 128)].copy()
subset_summary = rand.groupby(["dataset","horizon"], as_index=False).agg(
    n_subset_seeds=("target_seed","nunique"),
    mean_pooled_gain_pct=("pooled_gain_pct","mean"),
    std_pooled_gain_pct=("pooled_gain_pct","std"),
    median_pooled_gain_pct=("pooled_gain_pct","median"),
    min_pooled_gain_pct=("pooled_gain_pct","min"),
    max_pooled_gain_pct=("pooled_gain_pct","max"),
    positive_seed_fraction=("pooled_gain_pct", lambda s: float((np.asarray(s)>0).mean())),
)
subset_summary.to_csv(ROOT / "full_POOLED_target_subset_robustness_summary.csv", index=False)
display(subset_summary.round(4))


## 4. Candidate-cap robustness — primary pooled metric


In [ ]:
caps = pooled[pooled["target_strategy"] == "even"].copy()
cap_table = caps.pivot_table(index=["dataset","horizon"], columns="candidate_cap", values="pooled_gain_pct").reset_index()
cap_table.to_csv(ROOT / "full_POOLED_candidate_cap_gain_table.csv", index=False)
display(cap_table.round(4))
